# Full Pipeline — All-QI KNN Anonymization

Self-contained batch notebook. **No separate target column** — every quasi-identifier (including binary columns like `churn`) is used in KNN distance and synthesized the same way.

Set `DATA_FILE`, parameter lists, and batch controls in **Configuration**; schema is inferred automatically when data is loaded.

Writes only `experiment_ranking.csv` in this folder.


In [ ]:
# --- imports ---
import itertools
import time
import tracemalloc
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, ks_2samp
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, RobustScaler, StandardScaler

ROW_SYNTHESIS_MODE = "donor"
MISSING_LABEL = "Missing"
TVD_THRESHOLD = 0.10
KS_THRESHOLD = 0.10
PASS_RATE_TARGET = 0.85
RANDOM_STATE = 42
EXPERIMENT_RANKING_CSV = "experiment_ranking.csv"


## Configuration


In [ ]:
# Data
DATA_FILE = "../Bank Customer Churn Prediction.csv"

# Optional schema override (None = auto-detect from CSV)
ID_COLS_OVERRIDE = None

# Batch controls
BATCH_START = 0
BATCH_LIMIT = None  # None = run full grid
CHECKPOINT_EVERY = 25

# Parameter lists (full cartesian product; no separate target column in all-QI mode)
K_ANONYMITY = [3, 5]
K_NEIGHBORS = [3, 15]
CAT_GEN_METHOD = ["mode", "weighted_mode", "probability"]
NUM_GEN_METHOD = ["interpolation", "weighted_mean"]
SCALER_METHOD = ["standard", "robust"]
DISTANCE_MODE = ["weighted_sum"]
NUM_DISTANCE_METRIC = ["euclidean"]
MINKOWSKI_P = [3]
CAT_DISTANCE_METRIC = ["hamming"]
NUM_WEIGHT = [1.0]
CAT_WEIGHT = [1.0]
DISTANCE_PROFILE = ["balanced"]

PARAMETER_COMBINATIONS = pd.DataFrame([
    {
        "k_anonymity": k,
        "k_neighbors": kn,
        "cat_gen_method": cat,
        "num_gen_method": num,
        "scaler_method": scale,
        "distance_mode": dmode,
        "num_distance_metric": nmet,
        "minkowski_p": mp,
        "cat_distance_metric": cmet,
        "num_weight": nw,
        "cat_weight": cw,
        "distance_profile": dprof,
    }
    for k, kn, cat, num, scale, dmode, nmet, mp, cmet, nw, cw, dprof in itertools.product(
        K_ANONYMITY,
        K_NEIGHBORS,
        CAT_GEN_METHOD,
        NUM_GEN_METHOD,
        SCALER_METHOD,
        DISTANCE_MODE,
        NUM_DISTANCE_METRIC,
        MINKOWSKI_P,
        CAT_DISTANCE_METRIC,
        NUM_WEIGHT,
        CAT_WEIGHT,
        DISTANCE_PROFILE,
    )
])

print("Grid:", len(PARAMETER_COMBINATIONS), "configs")
PARAMETER_COMBINATIONS.head()


## Metrics


In [ ]:
PSI_THRESHOLD = 0.25
RARE_CATEGORY_THRESHOLD = 0.05


def calculate_psi(real_values, synth_values, bins: int = 10) -> float:
    real_values = pd.Series(real_values).dropna().astype(float)
    synth_values = pd.Series(synth_values).dropna().astype(float)
    if len(real_values) == 0 or len(synth_values) == 0:
        return float("nan")
    if real_values.nunique() <= 1:
        return 0.0 if synth_values.nunique() <= 1 and synth_values.iloc[0] == real_values.iloc[0] else float("nan")
    quantiles = np.linspace(0, 1, bins + 1)
    edges = np.unique(np.quantile(real_values, quantiles))
    if len(edges) < 3:
        return float("nan")
    real_counts, _ = np.histogram(real_values, bins=edges)
    synth_counts, _ = np.histogram(synth_values, bins=edges)
    real_pct = real_counts / max(1, real_counts.sum())
    synth_pct = synth_counts / max(1, synth_counts.sum())
    epsilon = 1e-6
    return float(np.sum((synth_pct - real_pct) * np.log((synth_pct + epsilon) / (real_pct + epsilon))))


def categorical_distribution_metrics(real_data, synth_data, cat_cols):
    rows = []
    for col in cat_cols:
        real_dist = real_data[col].astype(str).value_counts(normalize=True, dropna=False)
        synth_dist = synth_data[col].astype(str).value_counts(normalize=True, dropna=False)
        all_categories = sorted(set(real_dist.index) | set(synth_dist.index))
        real_aligned = real_dist.reindex(all_categories, fill_value=0)
        synth_aligned = synth_dist.reindex(all_categories, fill_value=0)
        drift = float(0.5 * np.abs(real_aligned - synth_aligned).sum())
        real_categories = set(real_dist.index)
        synth_categories = set(synth_dist.index)
        rows.append({
            "column": col,
            "TVD": round(drift, 4),
            "pass_tvd": drift < TVD_THRESHOLD,
            "real_category_count": len(real_categories),
            "synthetic_category_count": len(synth_categories),
            "category_coverage": round(len(real_categories & synth_categories) / max(1, len(real_categories)), 4),
            "new_synthetic_category_count": len(synth_categories - real_categories),
            "missing_synthetic_category_count": len(real_categories - synth_categories),
            "max_category_proportion_diff": round(float(np.abs(real_aligned - synth_aligned).max()), 4),
        })
    return pd.DataFrame(rows)


def rare_category_metrics(original_df, synthetic_df, cols, threshold=RARE_CATEGORY_THRESHOLD):
    rows = []
    for col in cols:
        orig_freq = original_df[col].value_counts(normalize=True)
        rare_cats = orig_freq[orig_freq < threshold].index
        if len(rare_cats) == 0:
            rows.append({
                "column": col,
                "rare_category_count": 0,
                "original_rare_mass": 0.0,
                "synthetic_rare_mass": 0.0,
                "retention_ratio": float("nan"),
            })
            continue
        synth_freq = synthetic_df[col].value_counts(normalize=True)
        orig_mass = float(orig_freq.loc[rare_cats].sum())
        synth_mass = float(synth_freq.reindex(rare_cats, fill_value=0).sum())
        rows.append({
            "column": col,
            "rare_category_count": int(len(rare_cats)),
            "original_rare_mass": round(orig_mass, 4),
            "synthetic_rare_mass": round(synth_mass, 4),
            "retention_ratio": round(synth_mass / orig_mass, 4) if orig_mass > 0 else float("nan"),
        })
    return pd.DataFrame(rows)


def numerical_distribution_metrics(real_data, synth_data, num_cols):
    rows = []
    for col in num_cols:
        real = pd.to_numeric(real_data[col], errors="coerce").dropna()
        synth = pd.to_numeric(synth_data[col], errors="coerce").dropna()
        if len(real) == 0 or len(synth) == 0:
            continue
        ks_stat = float(ks_2samp(real, synth).statistic) if real.nunique() > 1 and synth.nunique() > 1 else float("nan")
        psi = calculate_psi(real, synth)
        rows.append({
            "column": col,
            "KS_statistic": round(ks_stat, 4) if pd.notna(ks_stat) else np.nan,
            "pass_ks": ks_stat < KS_THRESHOLD if pd.notna(ks_stat) else False,
            "psi": round(psi, 4) if pd.notna(psi) else np.nan,
            "pass_psi": psi < PSI_THRESHOLD if pd.notna(psi) else False,
            "real_mean": round(float(real.mean()), 4),
            "synthetic_mean": round(float(synth.mean()), 4),
            "mean_abs_diff": round(abs(float(real.mean() - synth.mean())), 4),
            "real_median": round(float(real.median()), 4),
            "synthetic_median": round(float(synth.median()), 4),
            "median_abs_diff": round(abs(float(real.median() - synth.median())), 4),
            "real_std": round(float(real.std()), 4),
            "synthetic_std": round(float(synth.std()), 4),
            "std_abs_diff": round(abs(float(real.std() - synth.std())), 4),
        })
    return pd.DataFrame(rows)


def cramers_v(x, y) -> float:
    tbl = pd.crosstab(x.astype(str), y.astype(str))
    if tbl.shape[0] < 2 or tbl.shape[1] < 2:
        return float("nan")
    chi2 = chi2_contingency(tbl)[0]
    n = tbl.sum().sum()
    k = min(tbl.shape) - 1
    return float(np.sqrt(chi2 / (n * k))) if n > 0 and k > 0 else 0.0


def numerical_correlation_metrics(real_data, synth_data, num_cols):
    if len(num_cols) < 2:
        return pd.DataFrame()
    real_corr = real_data[num_cols].astype(float).corr()
    synth_corr = synth_data[num_cols].astype(float).corr()
    rows = []
    for c1, c2 in itertools.combinations(num_cols, 2):
        real_val = real_corr.loc[c1, c2]
        synth_val = synth_corr.loc[c1, c2]
        rows.append({
            "feature_1": c1,
            "feature_2": c2,
            "real_corr": round(float(real_val), 4) if pd.notna(real_val) else np.nan,
            "synthetic_corr": round(float(synth_val), 4) if pd.notna(synth_val) else np.nan,
            "abs_corr_diff": round(abs(float(real_val) - float(synth_val)), 4)
            if pd.notna(real_val) and pd.notna(synth_val) else np.nan,
        })
    return pd.DataFrame(rows)


def categorical_pair_relationship_metrics(real_data, synth_data, cat_cols):
    rows = []
    for c1, c2 in itertools.combinations(cat_cols, 2):
        real_v = cramers_v(real_data[c1], real_data[c2])
        synth_v = cramers_v(synth_data[c1], synth_data[c2])
        rows.append({
            "feature_1": c1,
            "feature_2": c2,
            "real_cramers_v": round(real_v, 4) if pd.notna(real_v) else np.nan,
            "synthetic_cramers_v": round(synth_v, 4) if pd.notna(synth_v) else np.nan,
            "abs_cramers_v_diff": round(abs(real_v - synth_v), 4) if pd.notna(real_v) and pd.notna(synth_v) else np.nan,
        })
    return pd.DataFrame(rows)


def categorical_numerical_relationship_metrics(real_data, synth_data, cat_cols, num_cols, max_categories=25):
    rows = []
    for cat in cat_cols:
        if real_data[cat].nunique(dropna=False) > max_categories:
            continue
        for num in num_cols:
            real_group = real_data.groupby(cat)[num].mean()
            synth_group = synth_data.groupby(cat)[num].mean()
            common_groups = sorted(set(real_group.index) & set(synth_group.index))
            if not common_groups:
                continue
            real_values = real_group.reindex(common_groups)
            synth_values = synth_group.reindex(common_groups)
            scale = real_data[num].astype(float).std()
            if pd.isna(scale) or scale == 0:
                scale = 1.0
            mean_group_diff = float(np.mean(np.abs(real_values - synth_values)) / scale)
            rows.append({
                "categorical_column": cat,
                "numerical_column": num,
                "common_category_count": len(common_groups),
                "normalised_group_mean_diff": round(mean_group_diff, 4),
            })
    return pd.DataFrame(rows)


def privacy_metrics(df_actual, df_out, qi_cols, suppressed):
    full_hash_actual = pd.util.hash_pandas_object(df_actual[qi_cols].astype(str), index=False)
    full_hash_out = pd.util.hash_pandas_object(df_out[qi_cols].astype(str), index=False)
    replaced_idx = suppressed[suppressed].index
    replaced_out = df_out.loc[replaced_idx, qi_cols].astype(str)
    replaced_hash = pd.util.hash_pandas_object(replaced_out, index=False)
    synthetic_duplicate_rate = 1.0 - (replaced_hash.nunique() / max(1, len(replaced_hash)))
    exact_match_to_actual_rate = float(replaced_hash.isin(set(full_hash_actual)).mean())
    full_dataset_duplicate_rate = 1.0 - (full_hash_out.nunique() / max(1, len(full_hash_out)))
    replaced_feature_cols = [c for c in df_actual.columns if c not in set(globals().get("ID_COLS", []))]
    exact_match_replaced_rows = len(
        df_out.loc[replaced_idx, replaced_feature_cols].merge(df_actual[replaced_feature_cols], how="inner")
    ) / max(len(replaced_idx), 1)
    return pd.DataFrame([{
        "replaced_rows": int(len(replaced_idx)),
        "replaced_unique_profiles": int(replaced_hash.nunique()),
        "synthetic_duplicate_rate": round(synthetic_duplicate_rate, 6),
        "exact_match_to_actual_rate": round(exact_match_to_actual_rate, 6),
        "full_dataset_duplicate_rate": round(full_dataset_duplicate_rate, 6),
        "replaced_exact_match_rate": round(exact_match_replaced_rows, 6),
    }])


def compute_all_qi_metrics(df_actual, df_out, suppressed):
    category_metrics = categorical_distribution_metrics(df_actual, df_out, TVD_CATEGORICAL_COLS)
    numeric_metrics = numerical_distribution_metrics(df_actual, df_out, NUMERICAL_COLS)
    rare_category = rare_category_metrics(df_actual, df_out, TVD_CATEGORICAL_COLS)
    num_correlation_metrics = numerical_correlation_metrics(df_actual, df_out, NUMERICAL_COLS)
    cat_pair_metrics = categorical_pair_relationship_metrics(df_actual, df_out, CATEGORICAL_COLS)
    cat_num_metrics = categorical_numerical_relationship_metrics(
        df_actual, df_out, CATEGORICAL_COLS, NUMERICAL_COLS
    )
    privacy = privacy_metrics(df_actual, df_out, QI_COLS, suppressed)

    tvd_pass_rate = float(category_metrics["pass_tvd"].mean())
    ks_pass_rate = float(numeric_metrics["pass_ks"].mean())
    mean_psi = float(numeric_metrics["psi"].dropna().mean()) if len(numeric_metrics) else float("nan")
    mean_num_corr_drift = float(num_correlation_metrics["abs_corr_diff"].dropna().mean()) if len(num_correlation_metrics) else 0.0
    mean_cat_pair_drift = float(cat_pair_metrics["abs_cramers_v_diff"].dropna().mean()) if len(cat_pair_metrics) else 0.0
    mean_cat_num_drift = float(cat_num_metrics["normalised_group_mean_diff"].dropna().mean()) if len(cat_num_metrics) else 0.0
    mean_pair_drift = mean_cat_pair_drift + mean_num_corr_drift
    exact_match_rate = float(privacy["replaced_exact_match_rate"].iloc[0])

    karabo_summary = {
        "mean_psi": round(mean_psi, 4) if pd.notna(mean_psi) else None,
        "mean_num_corr_drift": round(mean_num_corr_drift, 4),
        "mean_cat_pair_drift": round(mean_cat_pair_drift, 4),
        "mean_cat_num_drift": round(mean_cat_num_drift, 4),
        "synthetic_duplicate_rate": float(privacy["synthetic_duplicate_rate"].iloc[0]),
        "replaced_exact_match_rate": exact_match_rate,
    }

    scorecard = pd.DataFrame([
        {"area": "Quality-Categorical", "metric": "tvd_pass_rate>=0.85", "value": round(tvd_pass_rate, 4), "pass": tvd_pass_rate >= PASS_RATE_TARGET},
        {"area": "Quality-Numerical", "metric": "ks_pass_rate>=0.85", "value": round(ks_pass_rate, 4), "pass": ks_pass_rate >= PASS_RATE_TARGET},
        {"area": "Quality-Numerical", "metric": "mean_KS<0.10", "value": round(numeric_metrics["KS_statistic"].mean(), 4), "pass": numeric_metrics["KS_statistic"].mean() < KS_THRESHOLD},
        {"area": "Relationships", "metric": "mean_pairwise_drift<0.10", "value": round(mean_pair_drift, 4), "pass": mean_pair_drift < 0.10},
        {"area": "Privacy", "metric": "replaced_exact_match_rate<0.001", "value": round(exact_match_rate, 6), "pass": exact_match_rate < 0.001},
        {"area": "Suppression", "metric": "recovery_rate", "value": 1.0, "pass": suppressed.sum() > 0},
    ])
    overall_pass = bool(scorecard["pass"].all())
    distribution_summary = build_distribution_summary(df_actual, df_out)

    return {
        "category_metrics": category_metrics,
        "numeric_metrics": numeric_metrics,
        "rare_category_metrics": rare_category,
        "num_correlation_metrics": num_correlation_metrics,
        "cat_pair_relationship_metrics": cat_pair_metrics,
        "cat_num_relationship_metrics": cat_num_metrics,
        "privacy_metrics": privacy,
        "karabo_summary": karabo_summary,
        "scorecard": scorecard,
        "distribution_summary": distribution_summary,
        "overall_pass": overall_pass,
        "tvd_pass_rate": round(tvd_pass_rate, 4),
        "ks_pass_rate": round(ks_pass_rate, 4),
        "mean_tvd": round(category_metrics["TVD"].mean(), 4),
        "mean_ks": round(numeric_metrics["KS_statistic"].mean(), 4),
        "mean_pair_drift": round(mean_pair_drift, 6),
        "exact_match_rate": round(exact_match_rate, 6),
    }


print("All-QI validation metrics loaded.")


## Pipeline functions


In [ ]:
ID_NAME_HINTS = ("id", "uuid", "key")
LOW_CARDINALITY_NUMERIC = 10


def infer_dataset_schema_all_qi(df: pd.DataFrame, id_cols_override=None):
    id_cols = list(id_cols_override or [])
    if not id_cols:
        for col in df.columns:
            lc = col.lower()
            if df[col].nunique(dropna=True) == len(df) and (
                any(h in lc for h in ID_NAME_HINTS) or lc.endswith("_id") or lc == "id"
            ):
                id_cols.append(col)

    pool = [c for c in df.columns if c not in set(id_cols)]
    categorical_cols, numerical_cols = [], []
    for col in pool:
        series = df[col]
        nunique = series.nunique(dropna=True)
        if pd.api.types.is_numeric_dtype(series) and nunique > LOW_CARDINALITY_NUMERIC:
            numerical_cols.append(col)
        else:
            categorical_cols.append(col)

    kanon_qi_cols = list(categorical_cols)
    for col in numerical_cols:
        if df[col].nunique(dropna=True) <= 100:
            kanon_qi_cols.append(col)

    generalization = {}
    for col in kanon_qi_cols:
        nums = pd.to_numeric(df[col], errors="coerce")
        if nums.notna().mean() < 0.5:
            continue
        nunique = nums.nunique(dropna=True)
        if nunique <= 1:
            continue
        if nunique > 30:
            generalization[col] = {"type": "quantile_bins", "q": 20}
        elif nunique > 8:
            span = float(nums.max() - nums.min())
            step = max(1, int(round(span / 20)))
            generalization[col] = {"type": "bin_floor", "step": step}

    tvd_categorical_cols = list(categorical_cols)
    qi_feature_cols = categorical_cols + numerical_cols
    qi_cols = list(qi_feature_cols)
    missing = [c for c in qi_cols if c not in df.columns]
    if missing:
        raise ValueError(f"inferred QI columns missing from data: {missing}")

    return {
        "id_cols": id_cols,
        "categorical_cols": categorical_cols,
        "numerical_cols": numerical_cols,
        "kanon_qi_cols": kanon_qi_cols,
        "generalization": generalization,
        "tvd_categorical_cols": tvd_categorical_cols,
        "qi_feature_cols": qi_feature_cols,
        "qi_cols": qi_cols,
    }


def apply_schema(schema: dict):
    global ID_COLS, CATEGORICAL_COLS, NUMERICAL_COLS
    global KANON_QI_COLS, GENERALIZATION, TVD_CATEGORICAL_COLS, QI_FEATURE_COLS, QI_COLS
    ID_COLS = schema["id_cols"]
    CATEGORICAL_COLS = schema["categorical_cols"]
    NUMERICAL_COLS = schema["numerical_cols"]
    KANON_QI_COLS = schema["kanon_qi_cols"]
    GENERALIZATION = schema["generalization"]
    TVD_CATEGORICAL_COLS = schema["tvd_categorical_cols"]
    QI_FEATURE_COLS = schema["qi_feature_cols"]
    QI_COLS = schema["qi_cols"]


def generalize_for_kanonymity(df, qi_cols):
    g = df[qi_cols].copy()
    for col, rule in GENERALIZATION.items():
        if col not in g.columns:
            continue
        if rule["type"] == "bin_floor":
            s = pd.to_numeric(g[col], errors="coerce")
            step = float(rule["step"])
            g[col] = (s // step) * step
        elif rule["type"] == "quantile_bins":
            ranked = pd.to_numeric(g[col], errors="coerce").rank(method="first")
            g[col] = pd.qcut(ranked, q=int(rule.get("q", 20)), duplicates="drop").astype(str)
    str_cols = g.select_dtypes(include=["object", "string"]).columns
    for col in str_cols:
        g[col] = g[col].astype(str).fillna(MISSING_LABEL)
    for col in CATEGORICAL_COLS:
        if col in g.columns:
            g[col] = g[col].astype(str).fillna(MISSING_LABEL)
    return g


def identify_suppressed(df, k_anonymity):
    generalized = generalize_for_kanonymity(df, KANON_QI_COLS)
    class_sizes = generalized.groupby(list(generalized.columns), dropna=False).transform("size")
    suppressed = class_sizes < k_anonymity
    pool_idx = np.where(~suppressed.values)[0]
    synth_idx = np.where(suppressed.values)[0]
    if len(pool_idx) == 0:
        raise ValueError(f"k_anonymity={k_anonymity}: no neighbour pool")
    return suppressed, pool_idx, synth_idx


def fit_preprocessors(df, scaler_method):
    cat_encoders = {}
    for col in CATEGORICAL_COLS:
        enc = LabelEncoder()
        enc.fit(df[col].astype(str).fillna(MISSING_LABEL))
        cat_encoders[col] = enc
    num_medians = df[NUMERICAL_COLS].median()
    num_filled = df[NUMERICAL_COLS].fillna(num_medians).values.astype(float)
    scaler_cls = {"standard": StandardScaler, "minmax": MinMaxScaler, "robust": RobustScaler}[scaler_method]
    num_scaler = scaler_cls().fit(num_filled)
    num_p01 = {c: np.percentile(df[c].dropna(), 1) for c in NUMERICAL_COLS}
    num_p99 = {c: np.percentile(df[c].dropna(), 99) for c in NUMERICAL_COLS}
    X_cat = np.column_stack([
        cat_encoders[c].transform(df[c].astype(str).fillna(MISSING_LABEL)) for c in CATEGORICAL_COLS
    ]) if CATEGORICAL_COLS else np.empty((len(df), 0))
    X_num = num_scaler.transform(num_filled)
    num_ranges = np.ptp(num_filled, axis=0)
    return {
        "cat_encoders": cat_encoders,
        "num_medians": num_medians,
        "num_scaler": num_scaler,
        "num_p01": num_p01,
        "num_p99": num_p99,
        "X_cat": X_cat,
        "X_num": X_num,
        "num_filled": num_filled,
        "num_ranges": num_ranges,
    }


def categorical_pairwise_distance(pool_cat, synth_cat, metric):
    mismatch = pool_cat[np.newaxis, :, :] != synth_cat[:, np.newaxis, :]
    n_cat = pool_cat.shape[1]
    if n_cat == 0:
        return np.zeros((len(synth_cat), len(pool_cat)))
    matches = (~mismatch).sum(axis=2).astype(float)
    if metric == "hamming":
        return np.mean(mismatch, axis=2)
    if metric == "jaccard":
        union = 2.0 * n_cat - matches
        with np.errstate(divide="ignore", invalid="ignore"):
            sim = np.where(union > 0, matches / union, 1.0)
        return 1.0 - sim
    if metric == "overlap":
        return 1.0 - matches / n_cat
    raise ValueError(f"Unknown cat distance: {metric}")


def build_neighbor_cache(cfg, prep, pool_idx, synth_idx):
    pool_cat = prep["X_cat"][pool_idx]
    synth_cat = prep["X_cat"][synth_idx]
    if cfg["distance_mode"] == "gower":
        pool_num = prep["num_filled"][pool_idx]
        synth_num = prep["num_filled"][synth_idx]
        n_synth, n_pool = len(synth_num), len(pool_num)
        total_dist = np.zeros((n_synth, n_pool))
        safe_ranges = np.where(prep["num_ranges"] < 1e-8, 1.0, prep["num_ranges"])
        for j in range(pool_num.shape[1]):
            total_dist += np.abs(pool_num[np.newaxis, :, j] - synth_num[:, np.newaxis, j]) / safe_ranges[j]
        for j in range(pool_cat.shape[1]):
            total_dist += (pool_cat[np.newaxis, :, j] != synth_cat[:, np.newaxis, j]).astype(float)
        total_dist /= max(1, pool_num.shape[1] + pool_cat.shape[1])
    else:
        pool_num = prep["X_num"][pool_idx]
        synth_num = prep["X_num"][synth_idx]
        diff = pool_num[np.newaxis, :, :] - synth_num[:, np.newaxis, :]
        metric = cfg["num_distance_metric"]
        if metric == "euclidean":
            num_dist = np.linalg.norm(diff, axis=2)
        elif metric == "manhattan":
            num_dist = np.sum(np.abs(diff), axis=2)
        elif metric == "minkowski":
            p = int(cfg["minkowski_p"])
            num_dist = np.sum(np.abs(diff) ** p, axis=2) ** (1.0 / p)
        else:
            raise ValueError(metric)
        cat_dist = categorical_pairwise_distance(pool_cat, synth_cat, cfg["cat_distance_metric"])
        total_dist = float(cfg["num_weight"]) * num_dist + float(cfg["cat_weight"]) * cat_dist

    k_cache = min(int(cfg["k_neighbors"]), len(pool_idx))
    cache = {}
    for local_i, global_i in enumerate(synth_idx):
        row_dist = total_dist[local_i]
        idx_local = np.argpartition(row_dist, k_cache - 1)[:k_cache]
        order = idx_local[np.argsort(row_dist[idx_local])]
        cache[int(global_i)] = (row_dist[order], pool_idx[order])
    return cache


def generate_categorical(vals, weights, method, rng):
    if method == "mode":
        return Counter(vals).most_common(1)[0][0]
    if method == "weighted_mode":
        counts = defaultdict(float)
        for v, w in zip(vals, weights):
            counts[str(v)] += w
        return max(counts, key=counts.get)
    if method == "probability":
        return str(rng.choice(vals, p=weights / weights.sum()))
    raise ValueError(method)


def _cast_column_value(col, val, df):
    cat_cols = set(CATEGORICAL_COLS)
    if col in cat_cols and col in df.columns and pd.api.types.is_integer_dtype(df[col]):
        return int(float(val)) if str(val).replace(".", "", 1).isdigit() else val
    if col in NUMERICAL_COLS and col in df.columns and pd.api.types.is_integer_dtype(df[col]):
        return int(round(val))
    return val


def synthesize_dataset(df, cfg, prep, suppressed, pool_idx, synth_idx, neighbor_cache):
    rng = np.random.default_rng(int(cfg.get("random_state", RANDOM_STATE)))
    df_out = df.copy()
    X_num = prep["X_num"]

    for row_idx in synth_idx:
        row_idx = int(row_idx)
        dists_full, nbrs_full = neighbor_cache[row_idx]
        k = min(int(cfg["k_neighbors"]), len(dists_full))
        dists, nbrs = dists_full[:k], nbrs_full[:k]
        w = 1.0 / (dists + 1e-8)
        w = w / w.sum()
        row_out = {}
        synthesis_mode = str(cfg.get("row_synthesis_mode", ROW_SYNTHESIS_MODE)).lower()

        if synthesis_mode == "donor":
            neighbour_pool = df.loc[nbrs]
            for col in CATEGORICAL_COLS:
                vals = neighbour_pool[col].astype(str).fillna(MISSING_LABEL).values
                row_out[col] = str(rng.choice(vals, p=w))
            synth_scaled = np.zeros(len(NUMERICAL_COLS))
            donor = int(rng.choice(nbrs, p=w))
            for j in range(len(NUMERICAL_COLS)):
                t = float(rng.uniform(0.1, 0.9))
                synth_scaled[j] = X_num[row_idx, j] + t * (X_num[donor, j] - X_num[row_idx, j])
        else:
            for col in CATEGORICAL_COLS:
                vals = df.loc[nbrs, col].astype(str).fillna(MISSING_LABEL).values
                row_out[col] = generate_categorical(vals, w, cfg["cat_gen_method"], rng)
            synth_scaled = np.zeros(len(NUMERICAL_COLS))
            for j in range(len(NUMERICAL_COLS)):
                if cfg["num_gen_method"] == "interpolation":
                    nj = rng.choice(nbrs)
                    t = rng.random()
                    synth_scaled[j] = X_num[row_idx, j] + t * (X_num[nj, j] - X_num[row_idx, j])
                elif cfg["num_gen_method"] == "weighted_mean":
                    synth_scaled[j] = float(np.dot(w, X_num[nbrs, j]))
                else:
                    raise ValueError(cfg["num_gen_method"])

        synth_num = prep["num_scaler"].inverse_transform(synth_scaled.reshape(1, -1)).flatten()
        for j, col in enumerate(NUMERICAL_COLS):
            row_out[col] = float(np.clip(synth_num[j], prep["num_p01"][col], prep["num_p99"][col]))

        for col in QI_COLS:
            df_out.loc[row_idx, col] = _cast_column_value(col, row_out[col], df)
    return df_out


def is_valid_config(cfg):
    if cfg["distance_mode"] == "gower":
        if cfg["cat_distance_metric"] != "hamming":
            return False
        if cfg["scaler_method"] != "minmax":
            return False
        if cfg["num_distance_metric"] != "euclidean":
            return False
        if float(cfg["num_weight"]) != 1.0 or float(cfg["cat_weight"]) != 1.0:
            return False
    if cfg["num_distance_metric"] != "minkowski" and int(cfg["minkowski_p"]) != 3:
        return False
    return True


def config_to_key(cfg):
    k = int(cfg["k_anonymity"])
    mode = cfg["distance_mode"]
    if mode == "gower":
        return (k, mode)
    return (
        k, cfg["scaler_method"], mode,
        cfg["num_distance_metric"], cfg["cat_distance_metric"], int(cfg["minkowski_p"]),
        float(cfg["num_weight"]), float(cfg["cat_weight"]),
    )


def make_folder_name(idx, cfg):
    cat_tag = f"-cat-{cfg['cat_distance_metric']}" if cfg["distance_mode"] == "weighted_sum" else ""
    return (
        f"{idx:03d}_k{int(cfg['k_anonymity'])}_kn{int(cfg['k_neighbors'])}_cat-{cfg['cat_gen_method']}_"
        f"num-{cfg['num_gen_method']}_scale-{cfg['scaler_method']}_{cfg['distance_mode']}-"
        f"{cfg['num_distance_metric']}{cat_tag}_w-{cfg['distance_profile']}"
    )


def make_display_name(cfg):
    cat_m = cfg["cat_distance_metric"] if cfg["distance_mode"] == "weighted_sum" else "gower"
    return (
        f"k={int(cfg['k_anonymity'])} kn={int(cfg['k_neighbors'])} "
        f"cat={cfg['cat_gen_method']} num={cfg['num_gen_method']} "
        f"scale={cfg['scaler_method']} {cfg['distance_mode']}/{cfg['num_distance_metric']}/{cat_m}/{cfg['distance_profile']}"
    )


def finalize_ranking(rows):
    ranking = pd.DataFrame(rows)
    if ranking.empty:
        ranking["rank"] = []
        return ranking

    relationship_score = 1.0 - (ranking["mean_pair_drift"] / 0.10).clip(lower=0.0, upper=1.0)
    ks_quality = 1.0 - (ranking["mean_ks"] / 0.10).clip(lower=0.0, upper=1.0)
    ranking["relationship_score"] = relationship_score.round(4)
    ranking["ks_quality"] = ks_quality.round(4)
    ranking["quality_score"] = (
        0.33 * ranking["tvd_pass_rate"]
        + 0.33 * ranking["ks_pass_rate"]
        + 0.20 * relationship_score
        + 0.14 * ks_quality
    ).round(4)
    runtime_norm = ranking["runtime_sec"] / max(float(ranking["runtime_sec"].max()), 1e-6)
    ranking["composite_score"] = (ranking["quality_score"] - 0.02 * runtime_norm).round(4)
    ranking = ranking.sort_values(
        by=["overall_pass", "composite_score", "mean_pair_drift", "runtime_sec"],
        ascending=[False, False, True, True],
    ).reset_index(drop=True)
    ranking["rank"] = ranking.index + 1
    return ranking


def row_to_config(row):
    cfg = {
        "k_anonymity": int(row["k_anonymity"]),
        "k_neighbors": int(row["k_neighbors"]),
        "cat_gen_method": str(row["cat_gen_method"]),
        "num_gen_method": str(row["num_gen_method"]),
        "scaler_method": str(row["scaler_method"]),
        "distance_mode": str(row["distance_mode"]),
        "num_distance_metric": str(row["num_distance_metric"]),
        "minkowski_p": int(row["minkowski_p"]),
        "cat_distance_metric": str(row["cat_distance_metric"]),
        "num_weight": float(row["num_weight"]),
        "cat_weight": float(row["cat_weight"]),
        "distance_profile": str(row["distance_profile"]),
        "random_state": RANDOM_STATE,
        "row_synthesis_mode": str(row.get("row_synthesis_mode", ROW_SYNTHESIS_MODE)),
    }
    return cfg


def build_distribution_summary(df_actual, df_out):
    dist_rows = []
    for col in NUMERICAL_COLS:
        actual = pd.to_numeric(df_actual[col], errors="coerce")
        synth = pd.to_numeric(df_out[col], errors="coerce")
        actual_mean = float(actual.mean())
        synth_mean = float(synth.mean())
        dist_rows.append({
            "column": col,
            "type": "numerical",
            "actual_mean": round(actual_mean, 4),
            "synthetic_mean": round(synth_mean, 4),
            "actual_std": round(float(actual.std()), 4),
            "synthetic_std": round(float(synth.std()), 4),
            "mean_diff_pct": round(abs(synth_mean - actual_mean) / max(abs(actual_mean), 1e-6) * 100, 2),
        })
    for col in TVD_CATEGORICAL_COLS:
        if col not in df_actual.columns:
            continue
        actual_counts = df_actual[col].astype(str).value_counts(normalize=True)
        synth_counts = df_out[col].astype(str).value_counts(normalize=True)
        dist_rows.append({
            "column": col,
            "type": "categorical",
            "actual_category_count": int(actual_counts.shape[0]),
            "synthetic_category_count": int(synth_counts.shape[0]),
            "actual_top_category_pct": round(float(actual_counts.iloc[0]) * 100, 2),
            "synthetic_top_category_pct": round(float(synth_counts.iloc[0]) * 100, 2),
            "mean_diff_pct": np.nan,
        })
    return pd.DataFrame(dist_rows)

print("Pipeline functions loaded.")


## Load data


In [ ]:
df = pd.read_csv(DATA_FILE)
df = df.replace(r"^\s*$", np.nan, regex=True)
df = df.drop(columns=[c for c in df.columns if df[c].isna().all()]).reset_index(drop=True)

schema = infer_dataset_schema_all_qi(df, ID_COLS_OVERRIDE)
apply_schema(schema)

print("Loaded", f"{len(df):,}", "rows from", DATA_FILE)
print("ID columns (excluded):", ID_COLS)
print("Categorical QI:", CATEGORICAL_COLS)
print("Numerical QI:", NUMERICAL_COLS)
print("TVD categorical cols:", TVD_CATEGORICAL_COLS)
print("QI feature cols:", QI_FEATURE_COLS)
print("QI cols:", QI_COLS)
print("Row synthesis mode:", ROW_SYNTHESIS_MODE)
print("k-anonymity columns:", KANON_QI_COLS)
print("Generalization rules:", GENERALIZATION)
pd.DataFrame({
    "role": ["id", "categorical", "numerical", "tvd_categorical", "qi_features", "k-anon"],
    "columns": [
        ID_COLS,
        CATEGORICAL_COLS,
        NUMERICAL_COLS,
        TVD_CATEGORICAL_COLS,
        QI_FEATURE_COLS,
        KANON_QI_COLS,
    ],
})


## Run experiments


In [ ]:
suppression_cache = {}
prep_cache = {}
neighbor_cache_store = {}
ranking_rows = []

combos = PARAMETER_COMBINATIONS
end = len(combos) if BATCH_LIMIT is None else min(len(combos), BATCH_START + BATCH_LIMIT)
combos = combos.iloc[BATCH_START:end].reset_index(drop=True)
print("Running", len(combos), "configs")

for i, row in combos.iterrows():
    cfg = row_to_config(row)
    folder = make_folder_name(i + 1 + BATCH_START, cfg)
    print(f"[{i+1}/{len(combos)}] {folder}")

    tracemalloc.start()
    t0 = time.perf_counter()

    k = int(cfg["k_anonymity"])
    if k not in suppression_cache:
        suppression_cache[k] = identify_suppressed(df, k)
    suppressed, pool_idx, synth_idx = suppression_cache[k]

    scaler = cfg["scaler_method"]
    if scaler not in prep_cache:
        prep_cache[scaler] = fit_preprocessors(df, scaler)
    prep = prep_cache[scaler]

    nkey = config_to_key(cfg)
    if nkey not in neighbor_cache_store:
        neighbor_cache_store[nkey] = build_neighbor_cache(cfg, prep, pool_idx, synth_idx)
    neighbor_cache = neighbor_cache_store[nkey]

    df_out = synthesize_dataset(df, cfg, prep, suppressed, pool_idx, synth_idx, neighbor_cache)
    metrics = compute_all_qi_metrics(df, df_out, suppressed)

    runtime_sec = round(time.perf_counter() - t0, 3)
    _, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    ranking_rows.append({
        "folder": folder,
        "name": make_display_name(cfg),
        "k_anonymity": k,
        "k_neighbors": int(cfg["k_neighbors"]),
        "cat_gen_method": cfg["cat_gen_method"],
        "num_gen_method": cfg["num_gen_method"],
        "scaler_method": scaler,
        "num_weight": float(cfg["num_weight"]),
        "cat_weight": float(cfg["cat_weight"]),
        "distance_profile": cfg["distance_profile"],
        "distance_mode": cfg["distance_mode"],
        "num_distance_metric": cfg["num_distance_metric"],
        "cat_distance_metric": cfg["cat_distance_metric"],
        "minkowski_p": int(cfg["minkowski_p"]),
        "random_state": int(cfg.get("random_state", RANDOM_STATE)),
        "runtime_sec": runtime_sec,
        "peak_memory_mb": round(peak_mem / 1024 / 1024, 2),
        "n_suppressed": int(suppressed.sum()),
        "tvd_pass_rate": metrics["tvd_pass_rate"],
        "ks_pass_rate": metrics["ks_pass_rate"],
        "mean_tvd": metrics["mean_tvd"],
        "mean_ks": metrics["mean_ks"],
        "mean_pair_drift": metrics["mean_pair_drift"],
        "exact_match_rate": metrics["exact_match_rate"],
        "mean_psi": metrics.get("karabo_summary", {}).get("mean_psi"),
        "overall_pass": metrics["overall_pass"],
    })

    if len(ranking_rows) % CHECKPOINT_EVERY == 0:
        finalize_ranking(ranking_rows).to_csv(EXPERIMENT_RANKING_CSV, index=False)
        print("  checkpoint ->", EXPERIMENT_RANKING_CSV)

ranking = finalize_ranking(ranking_rows)
ranking.to_csv(EXPERIMENT_RANKING_CSV, index=False)
print("\nSaved", len(ranking), "rows ->", EXPERIMENT_RANKING_CSV)
if len(ranking):
    print("Top:", ranking.iloc[0]["folder"], "| overall_pass=", ranking.iloc[0]["overall_pass"])
ranking.head(10)
